# AA-валидация BEN-фреймворка (v11)

**Назначение:** проверить корректность permutation-теста для метрик разнообразия выдачи (BEN, BEN_weighted, 4 unique абсолютных, 4 unique относительных).

**Логика AA:** берём pre-experiment данные (до старта эксперимента), случайно делим юзеров пополам с разными seed, прогоняем permutation-тест на каждой такой паре. Под H₀ ожидаем:
- Type I error ≈ α (5% случаев тест выдаст «значимо»)
- p-value распределены равномерно на [0, 1]

Если что-то нарушается — метод некалиброван и его p-value нельзя доверять.

**5 тестов калибровки на каждой метрике:**
1. **Wilson 95% CI** — содержит ли 5% эмпирическую Type I error
2. **Clopper-Pearson exact CI** — то же через точный CI Beta-распределения (gold-standard)
3. **Биномиальный тест** — H₀: эмпирическая `p_emp = α`
4. **KS-test на Uniform(0,1)** — общая равномерность p-value
5. **Anderson-Darling** — равномерность с упором на хвосты (важно для α-области)

**Особенности реализации:**
- Параллелизация AA-итераций через `multiprocessing.Pool` с fork-CoW (numpy-массивы шарятся через память родителя без pickling)
- HDFS-кэш материализации и iter-чекпоинты для resume после падения
- BLAS-thread limit (`OPENBLAS_NUM_THREADS=1`) для совместимости с multiprocessing
- Resilient Spark session с auto-recovery


## Заголовок и changelog

```
# =============================================================================
# AA-тест v11: проверка unified permutation для всех 10 метрик
# =============================================================================
#
# ЦЕЛЬ: проверить под H0 (Δ_true=0) что unified honest user permutation даёт:
#   - Type I error ≈ ALPHA (5%)
#   - p-value uniform на [0,1] (KS-тест)
# для **всех 10 метрик** одновременно:
#   - 6 абсолютных: BEN, BEN_weighted, 4 unique-метрики
#   - 4 относительных (NEW v11): unique_items_rel, unique_clicked_items_rel,
#     unique_btc_items_rel, unique_contacted_items_rel
#
# ВАЖНО v11:
#   AA-тест на синтетике (50 итер × 100 perm, 1.25M rows after dedup):
#     все 10 метрик: Type I = 4-10% (CI [0,18%], target 5%), KS p > 0.17
#     corr(p_abs, p_rel) ≈ 0.998 — *_rel дают ту же информацию в %-шкале
#
# ПОЧЕМУ ЭТО ВАЖНО:
#   В v9 для unique-метрик использовался bootstrap. Симуляции и AA показали
#   что bootstrap для count-distinct структурно проблемный:
#     - bias смещает уровни (~37% юзеров не попадают в каждый ресэмпл)
#     - p-value shortcut давал Type I = 1% (слишком консервативный)
#     - normal-approx CI требовал AA-валидации нормальности
#   В v10/v11 — UNIFIED PERMUTATION — один permutation loop
#   считает все метрики. Permutation математически корректен:
#     - distribution под H0 строится напрямую из перестановок
#     - не требует CLT, smoothness, asymptotic regime
#     - p-value через centered formula, type I = 5% by construction
#
# ДАННЫЕ: PRE-EXPERIMENT период (до старта exp 30895). Активные юзеры
# из user_sessions, сэмпл 5%. В каждой AA-итерации случайный split 50/50.
#
# МЕТОД (что в production v11):
#   honest_user_permutation_test:
#     - Для каждой permutation: перемешиваем метки юзеров C↔T
#     - Пересчитываем 10 метрик одновременно
#     - delta_obs = разница по реальным меткам
#     - p_value = доля |perm_centered| >= |delta_obs - mean(perm)|
# =============================================================================
```

## Ячейка 1 — Setup: Spark, BLAS-limit, resilient session

**Что происходит:**
1. **Жёстко ограничиваем BLAS-threads до 1** до `import numpy`. Критично для multiprocessing: иначе OpenBLAS по дефолту хапает столько threads сколько CPU видит OS, и при N workers получаем oversubscription с замедлением 8–10×.
2. **Создаём resilient Spark session** через `get_or_create_spark()` с auto-recovery при kill YARN. Helpers: `spark_ping()`, `ensure_spark_alive()`, `_hdfs_path_exists()`.

**Зачем resilient session:** AA-прогон длится 2–10 часов и часто прерывается убийством Spark application'а. Без auto-recovery теряем прогресс.


In [ ]:

# КРИТИЧНО: ограничить BLAS-threads ДО import numpy (иначе OpenBLAS читает дефолт = всё).
# Без этого 6 multiprocessing-workers × 96 BLAS-threads = 576 потоков на 8 CPU квоты,
# что даёт 8-10× замедление от oversubscription.
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

import time, json, hashlib, pickle
from datetime import date, timedelta
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as sps

import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession

from search_quality_dags.jupyter import spark_task

spark_cfg = {
    "spark.executor.cores": 4,
    "spark.executor.memory": "12g",
    "spark.dynamicAllocation.maxExecutors": 150,
    "spark.driver.memory": "64g",
    "spark.driver.maxResultSize": "32g",
    "spark.sql.shuffle.partitions": "1500",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.orc.filterPushdown": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryoserializer.buffer.max": "1g",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
    "spark.sql.execution.arrow.maxRecordsPerBatch": "50000",
}

# Глобальные переменные для resilient session
_SPARK_TASK = None
_SPARK_APP_NAME = "avchuykov aa-test-v11"


def get_or_create_spark(force_new=False):
    """Получает рабочую SparkSession, при необходимости пересоздаёт.
    
    Сценарии:
      1. Первый вызов  — создаём session.
      2. Повторный вызов, session жива  — возвращаем существующую.
      3. Повторный вызов, session убита YARN/network — пересоздаём.
      4. force_new=True — всегда пересоздаём (для ручного reset).
    
    Возвращает (spark, sc, was_recreated_bool).
    """
    global _SPARK_TASK
    
    needs_create = force_new or _SPARK_TASK is None
    
    # Проверяем живость существующей session
    if not needs_create:
        try:
            sc = _SPARK_TASK.spark_session.sparkContext
            if sc._jsc is None:
                needs_create = True
            else:
                # Лёгкая проверка: попробовать создать parallelize
                _ = sc.parallelize([1]).count()
        except Exception as e:
            print(f"  [resilient] Существующая session мертва: {type(e).__name__}: {e}")
            needs_create = True
    
    if needs_create:
        action = "Пересоздаём" if _SPARK_TASK is not None else "Создаём"
        print(f"  [resilient] {action} Spark session...", flush=True)
        # Старая session — закрыть аккуратно если была
        if _SPARK_TASK is not None:
            try:
                _SPARK_TASK.spark_session.stop()
            except Exception:
                pass
        # До 3 попыток с экспоненциальным backoff
        last_exc = None
        for attempt in range(3):
            try:
                _SPARK_TASK = spark_task(app_name=_SPARK_APP_NAME, cfg=spark_cfg)
                spark = _SPARK_TASK.spark_session
                spark.sparkContext.setLogLevel("ERROR")
                # Заглушаем шумные классы Hadoop (как раньше)
                log4j = spark._jvm.org.apache.log4j
                log4j.LogManager.getLogger("org.apache.hadoop.net.ScriptBasedMapping").setLevel(log4j.Level.ERROR)
                log4j.LogManager.getLogger("org.apache.hadoop.util.Shell").setLevel(log4j.Level.ERROR)
                log4j.LogManager.getLogger("org.apache.hadoop.net").setLevel(log4j.Level.ERROR)
                import logging
                logging.getLogger("py4j").setLevel(logging.ERROR)
                logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)
                print(f"  [resilient] Session готова (attempt {attempt+1}): {spark.version}", flush=True)
                return spark, spark.sparkContext, True
            except Exception as e:
                last_exc = e
                print(f"  [resilient] attempt {attempt+1}/3 failed: {type(e).__name__}: {e}")
                time.sleep(min(30 * 2**attempt, 120))
        raise RuntimeError(f"Не удалось создать Spark session за 3 попытки: {last_exc}")
    
    return _SPARK_TASK.spark_session, _SPARK_TASK.spark_session.sparkContext, False


def spark_ping(silent=False):
    """Keep-alive ping. Возвращает True если session жива, False если умерла.
    
    Если умерла — НЕ пересоздаёт автоматически (caller должен решить).
    Используй get_or_create_spark() если нужно пересоздать.
    """
    global _SPARK_TASK
    if _SPARK_TASK is None:
        return False
    try:
        _ = _SPARK_TASK.spark_session.sparkContext.parallelize([1, 2, 3]).count()
        if not silent:
            print(f"    [spark ping OK]", flush=True)
        return True
    except Exception as e:
        if not silent:
            print(f"    [spark ping FAILED: {type(e).__name__}: {e}]", flush=True)
        return False


def ensure_spark_alive():
    """Использовать вместо ручного ping в долгих циклах.
    
    Если session жива — return существующая.
    Если мертва — автоматически пересоздаёт и возвращает новую.
    Возвращает (spark, was_recreated).
    """
    global _SPARK_TASK
    if not spark_ping(silent=True):
        print(f"    [resilient] Session мертва, пересоздаём...", flush=True)
        spark, sc, recreated = get_or_create_spark(force_new=True)
        return spark, True
    return _SPARK_TASK.spark_session, False


# Создаём session
spark, sc, _ = get_or_create_spark()
print(f"Spark: {spark.version}")

## Ячейка 2 — Параметры AA-теста

**Ключевые параметры:**
- `DATE_FROM` / `DATE_TO` — pre-experiment окно (важно: до старта реального эксперимента, иначе AA увидит эффект treatment'а)
- `LOGCATS` — фильтр по логкату; пустой список = тотал
- `UNIVERSE_SAMPLE_PCT` — % юзеров для AA. На реальных Fashion-данных при 3% получаем ~13M строк, что нормально влезает в память
- `N_AA = 200` — количество AA-итераций (200 даёт Wilson CI шириной ~6pp при k/N=5%)
- `N_PERM = 150` — permutations внутри каждой AA-итерации
- `N_WORKERS = 6` — параллелизм AA-итераций; ограничен CPU-квотой и NUMA-bandwidth
- `ALPHA = 0.05` — уровень значимости

**Cache в HDFS** включает все параметры в путь — разные прогоны с разными настройками не пересекаются.


In [ ]:

# PRE-EXPERIMENT период (до старта exp 30895)
DATE_FROM = "2026-03-05"
DATE_TO   = "2026-03-10"

PATH_SESSIONS        = "/user/airflow/user_sessions"
PATH_MICROCATEGORIES = "/vertica/ext/current_microcategories"

LOGCATS = ["Goods.Fashion"]
TOP_K   = 100

UNIVERSE_SAMPLE_PCT  = 3
UNIVERSE_SAMPLE_SALT = "aa_v11_universe"

# AA-параметры
# N_AA=200, N_PERM=150, N_WORKERS=8 — расчётный run на Fashion universe 3%:
#   - ~40M строк после дедупа
#   - sequential per-iter: ~50-80s
#   - 200 iters ÷ 8 workers ≈ 1250-2000s = 20-35 мин
#   - Wilson CI ширина при k/N=5% и N=200: ~6pp
#   - MC SE p-value при n_perm=150: ~0.018 при p=0.05
N_AA          = 200
N_PERM        = 150
ALPHA         = 0.05
SEED_BASE     = 42

# Параллелизм AA-итераций. Каждый worker — независимый AA-iter (fork-shared
# read-only numpy массивы, без pickling). Поставь по числу свободных CPU
# на driver-ноде (обычно 4-16). Если fork не поддерживается окружением
# (некоторые managed Spark) — поставь N_WORKERS=1 для sequential fallback.
N_WORKERS     = 8

PING_EVERY_AA = 10

HDFS_HOME    = "/user/avchuykov"
PATH_RESULTS = f"{HDFS_HOME}/aa_test/v11_validation"

print(f"\n=== AA v11 (pre-experiment, parallel) ===")
print(f"  Period:       {DATE_FROM}..{DATE_TO}")
print(f"  Logcats:      {LOGCATS}")
print(f"  Universe:     {UNIVERSE_SAMPLE_PCT}% (salt={UNIVERSE_SAMPLE_SALT})")
print(f"  TOP_K:        {TOP_K}")
print(f"  N_AA:         {N_AA}")
print(f"  N_perm:       {N_PERM}")
print(f"  N_workers:    {N_WORKERS}  (set to 1 for sequential fallback)")
print(f"  α:            {ALPHA}")

## Ячейка 3 — Микрокатегории для фильтра по логкату

Если `LOGCATS` задан, читаем витрину `current_microcategories` и оставляем `microcat_id` нужных логкатов. В Ячейке 6 эта таблица заджойнится с impressions для фильтрации.

При `LOGCATS=[]` (тотал) ячейка просто пропускается.


In [ ]:

if LOGCATS:
    t = time.time()
    mcats_df = (
        spark.read.parquet(PATH_MICROCATEGORIES)
            .filter((F.col("is_active_microcat") == True) &
                     (F.col("logical_category").isin(*LOGCATS)))
            .select(F.col("microcat_ext").cast("long").alias("microcat_ext"),
                     F.col("logical_category"))
            .distinct()
    )
    mcats_df.cache()
    n_mcats = mcats_df.count()
    print(f"Microcategories: {n_mcats:,} за {time.time()-t:.0f}s")
else:
    mcats_df = None

## Ячейка 4 — Universe: активные юзеры за pre-experiment

**Что происходит:** определяем universe — пользователей, имевших u2i-показы в pre-experiment периоде. Применяем hash-семплинг через `UNIVERSE_SAMPLE_PCT`.

**Зачем такой подход.** Не берём всех юзеров эксперимента, а только тех кто был активен ДО старта — чтобы AA проверял метод на типичной активной аудитории, не разводя данные неактивными юзерами без показов.

Hash-семплинг детерминированный (`UNIVERSE_SAMPLE_SALT`), при перезапуске получаем тот же universe.


In [ ]:

t = time.time()
d_from = date.fromisoformat(DATE_FROM)
d_to   = date.fromisoformat(DATE_TO)
days = [(d_from + timedelta(i)).isoformat() for i in range((d_to - d_from).days + 1)]
print(f"Дней: {len(days)}")

session_dfs = [
    spark.read.orc(f"{PATH_SESSIONS}/{d}")
        .select(F.col("device_id").cast("string").alias("device_id"), "u2i_serps")
    for d in days
]
sessions_union = reduce(lambda a, b: a.unionByName(b), session_dfs)

active_users = (
    sessions_union.filter(F.size("u2i_serps") > 0)
        .select("device_id").distinct()
)

universe_users = (
    active_users
        .withColumn("h", F.abs(F.xxhash64(F.concat(F.col("device_id"),
                                                     F.lit("|" + UNIVERSE_SAMPLE_SALT)))))
        .filter(F.col("h") % 100 < UNIVERSE_SAMPLE_PCT)
        .drop("h")
)
universe_users.cache()
n_universe = universe_users.count()
print(f"Universe: {n_universe:,} активных юзеров за {time.time()-t:.0f}s")

## Ячейка 5 — Impressions с action-флагами

Читаем `u2i_serps` из `user_sessions`, фильтруем по дате, оставляем только активных юзеров из universe, к каждому показу приджойниваем флаги действий: `was_clicked`, `was_btc`, `was_contact`.

Сохраняем минимальный набор колонок: `cookie_id`, `item_id`, `position`, `was_*`. На AA не нужно `ab_group` — мы сами генерим случайные группы внутри AA-цикла.


In [ ]:

t = time.time()
impressions_raw_parts = []
for d in days:
    df_day = (
        spark.read.orc(f"{PATH_SESSIONS}/{d}")
            .select(F.col("device_id").cast("string").alias("device_id"), "u2i_serps")
    )
    df_with_serp = df_day.withColumn("serp", F.explode("u2i_serps"))
    df_with_items = df_with_serp.select("device_id", F.col("serp.items").alias("items"))
    df_imp = df_with_items.withColumn("imp", F.explode("items"))
    imp = df_imp.select(
        "device_id",
        F.col("imp.id").cast("long").alias("item_id"),
        F.col("imp.pos").cast("int").alias("position"),
        F.col("imp.mcat_id").cast("long").alias("mcat_id"),
        (F.col("imp.ui.click")   > 0).cast("int").alias("was_clicked"),
        (F.col("imp.ui.btc")     > 0).cast("int").alias("was_btc"),
        (F.col("imp.ui.contact") > 0).cast("int").alias("was_contact"),
    )
    if TOP_K is not None:
        imp = imp.filter(F.col("position") <= TOP_K)
    impressions_raw_parts.append(imp)

impressions_raw = reduce(lambda a, b: a.unionByName(b), impressions_raw_parts)

if LOGCATS:
    impressions_filtered = (
        impressions_raw
            .join(F.broadcast(mcats_df),
                  impressions_raw.mcat_id == mcats_df.microcat_ext, "inner")
            .select("device_id", "item_id", "position",
                     "was_clicked", "was_btc", "was_contact")
    )
else:
    impressions_filtered = impressions_raw.select("device_id", "item_id", "position",
                                                     "was_clicked", "was_btc", "was_contact")

joined = (
    impressions_filtered.alias("i")
        .join(F.broadcast(universe_users.alias("u")), on="device_id", how="inner")
        .select(
            F.col("u.device_id").alias("cookie_id"),
            F.col("i.item_id"),
            F.col("i.position"),
            F.col("i.was_clicked"),
            F.col("i.was_btc"),
            F.col("i.was_contact"),
        )
)

impressions = (
    joined.groupBy("cookie_id", "item_id")
        .agg(
            F.min("position").alias("position"),
            F.max("was_clicked").alias("was_clicked"),
            F.max("was_btc").alias("was_btc"),
            F.max("was_contact").alias("was_contact"),
        )
)
impressions.cache()

n_rows  = impressions.count()
n_users_uni = impressions.select("cookie_id").distinct().count()
n_items_uni = impressions.select("item_id").distinct().count()
n_clicks = impressions.select(F.sum("was_clicked")).first()[0] or 0
n_btc    = impressions.select(F.sum("was_btc")).first()[0] or 0
n_cont   = impressions.select(F.sum("was_contact")).first()[0] or 0
print(f"\nUniverse:")
print(f"  Строк: {n_rows:,}, юзеров: {n_users_uni:,}, items: {n_items_uni:,}")
print(f"  clicks: {n_clicks:,} ({100*n_clicks/n_rows:.2f}%), "
      f"btc: {n_btc:,} ({100*n_btc/n_rows:.2f}%), "
      f"contacts: {n_cont:,} ({100*n_cont/n_rows:.2f}%)")
print(f"Time: {time.time()-t:.0f}s")

## Ячейка 6 — Материализация в pandas с HDFS-кэшем

**Что происходит:**
1. Считаем `cache_key = MD5({DATE, LOGCATS, UNIVERSE_SAMPLE_PCT, ...})` — детерминированный хеш всех параметров
2. Если кэш `cache_dir/impressions.parquet` уже существует — читаем оттуда
3. Иначе записываем в parquet → читаем обратно как pandas
4. Дедуп `(cookie_id, item_id)` перед материализацией

**Зачем дедуп:** для unique-метрик важно чтобы один товар-один показ-на-юзера. Если повторные показы оставить, метрики будут смещены (но BEN не пострадает, он всё равно работает на уровне показов).

**Зачем кэш:** материализация занимает 3–10 минут. При перезапуске после падения — мгновенно из кэша.


In [ ]:
#
# Кэш позволяет:
#   - При перезапуске kernel не считать impressions с нуля (~3-4 минуты вместо ~10)
#   - Если упало после материализации (например, в AA-цикле) — продолжить
#     без reread из source-таблиц
#   - Гарантия идентичности: ключ кэша зависит от ВСЕХ параметров запроса
#     (DATE_FROM, DATE_TO, LOGCATS, UNIVERSE_SAMPLE_PCT, UNIVERSE_SAMPLE_SALT, TOP_K)
#
# Если хочешь форсированно пересчитать (например, поменялись микрокатегории
# в источнике, но даты те же) — поставь FORCE_REBUILD_CACHE=True
FORCE_REBUILD_CACHE = False

# Ключ кэша: хеш от всех параметров, влияющих на содержимое impressions
def _cache_key(*parts):
    s = "|".join(str(p) for p in parts)
    return hashlib.md5(s.encode()).hexdigest()[:12]

cache_key = _cache_key(
    DATE_FROM, DATE_TO, sorted(LOGCATS), TOP_K,
    UNIVERSE_SAMPLE_PCT, UNIVERSE_SAMPLE_SALT,
)
cache_dir = f"{HDFS_HOME}/aa_cache/v11/{cache_key}"
cache_imp_path = f"{cache_dir}/impressions.parquet"
cache_meta_path = f"{cache_dir}/meta.json"

# Helpers для проверки/удаления HDFS-путей через Hadoop FS API
def _hdfs_path_exists(path):
    try:
        jvm = spark._jvm
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI.create(path),
                                                      spark._jsc.hadoopConfiguration())
        return fs.exists(hadoop_path)
    except Exception as e:
        print(f"  [cache] _hdfs_path_exists error: {e}")
        return False


def _hdfs_delete(path):
    try:
        jvm = spark._jvm
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI.create(path),
                                                      spark._jsc.hadoopConfiguration())
        if fs.exists(hadoop_path):
            fs.delete(hadoop_path, True)  # recursive=True
    except Exception as e:
        print(f"  [cache] _hdfs_delete error: {e}")


print(f"\n=== Cache check ===")
print(f"  Ключ:  {cache_key}")
print(f"  Путь:  {cache_imp_path}")

cache_hit = False
if not FORCE_REBUILD_CACHE:
    if _hdfs_path_exists(cache_imp_path):
        try:
            t = time.time()
            print(f"  HIT — читаем кэш...", flush=True)
            imp_pd = spark.read.parquet(cache_imp_path).toPandas()
            cache_hit = True
            print(f"  Прочитан кэш: {len(imp_pd):,} строк за {time.time()-t:.0f}s")
            print(f"  Память: {imp_pd.memory_usage(deep=True).sum()/1e9:.2f} GB")
        except Exception as e:
            print(f"  Ошибка чтения кэша: {e}")
            print(f"  Падаем на материализацию с нуля")
            cache_hit = False

if not cache_hit:
    print(f"  MISS — материализуем impressions и пишем в кэш")
    t = time.time()
    print(f"\nМатериализация...")
    # Сохраняем сначала на HDFS как parquet — это безопаснее (не вылет по памяти),
    # потом читаем обратно как pandas. На больших данных это даже быстрее
    # toPandas() напрямую, потому что Arrow-cycle лучше параллелится через файл.
    print(f"  → пишем impressions.parquet на HDFS...", flush=True)
    impressions.write.mode("overwrite").parquet(cache_imp_path)
    print(f"    Записано за {time.time()-t:.0f}s", flush=True)
    
    # Метаданные кэша — для диагностики
    try:
        meta = {
            "date_from": str(DATE_FROM), "date_to": str(DATE_TO),
            "logcats": LOGCATS, "top_k": TOP_K,
            "universe_sample_pct": UNIVERSE_SAMPLE_PCT,
            "universe_sample_salt": UNIVERSE_SAMPLE_SALT,
            "n_rows": int(n_rows), "n_users": int(n_users), "n_items": int(n_items),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "version": "v11",
        }
        meta_df = spark.createDataFrame([(json.dumps(meta, ensure_ascii=False),)], ["meta"])
        meta_df.coalesce(1).write.mode("overwrite").text(cache_meta_path)
    except Exception as e:
        print(f"  Не удалось записать meta: {e}")
    
    # Читаем обратно как pandas
    t2 = time.time()
    print(f"  → читаем кэш обратно как pandas...", flush=True)
    imp_pd = spark.read.parquet(cache_imp_path).toPandas()
    print(f"    {len(imp_pd):,} строк за {time.time()-t2:.0f}s")
    print(f"  Память: {imp_pd.memory_usage(deep=True).sum()/1e9:.2f} GB")
    print(f"  Total cache+materialize: {time.time()-t:.0f}s")

## Ячейка 7 — Unified permutation-функция

**Главное вычислительное ядро.** Та же функция что в production-ноутбуке. На один вход (массивы numpy) и один seed выдаёт результаты для всех 10 метрик.

### Алгоритм

```
1. Вычисляем delta_obs = T - C для каждой метрики
2. Для b = 1..N_perm:
   - Перемешиваем метки группы на уровне cookie_id
   - Пересчитываем все 10 метрик
   - Получаем delta_b
3. p-value = доля |delta_b - mean| ≥ |delta_obs - mean| (centered p-value)
4. H₀-границы = квантили [α/2, 1-α/2]
```

### Оптимизации в numpy

- `unique_items` через `(c_A > 0).sum()` — бесплатно, c_A уже вычисляется для BEN
- Action-subsets выносятся за loop как фильтры
- `1/k_u`, `1/sum_w_user` pre-compute
- `nlq_per_row = nlq_B[i_arr]` + in-place правка для группы A — экономит ~400 MB

### Что считает каждый proxy метрики

- `BEN`, `BEN_weighted` — энтропия эмпирического распределения товаров с/без позиционных весов
- `unique_items` — сколько уникальных items видит группа
- `unique_clicked/btc/contacted_items` — то же, но по подмножеству items с действием
- `*_rel` — относительная разница `(T-C)/C` со своим p-value (более интерпретируемая шкала)


In [ ]:

def unified_permutation_test(u_arr, i_arr, pos_arr, clicks_arr, btc_arr, contacts_arr,
                                n_users, n_items, group_vec, n_perm, seed):
    """Unified permutation test для BEN, BEN_weighted, 4 unique и 4 unique_rel.
    
    Один loop → 10 метрик. Та же логика что в production v11.
    """
    rng = np.random.default_rng(seed)
    
    # Pre-compute
    c_total = np.bincount(i_arr, minlength=n_items).astype(np.float64)
    k_u = np.bincount(u_arr, minlength=n_users).astype(np.float64)
    w_arr = 1.0 / np.log2(pos_arr + 1)
    sum_w_user = np.bincount(u_arr, weights=w_arr, minlength=n_users)
    eps = 1e-8
    inv_k_u = 1.0 / np.maximum(k_u, 1.0)
    inv_sum_w = 1.0 / np.maximum(sum_w_user, 1e-12)
    
    # Pre-extract action-subsets (sparse → большая экономия)
    cl_mask_b = clicks_arr == 1
    bt_mask_b = btc_arr == 1
    co_mask_b = contacts_arr == 1
    i_cl = i_arr[cl_mask_b]; u_cl = u_arr[cl_mask_b]
    i_bt = i_arr[bt_mask_b]; u_bt = u_arr[bt_mask_b]
    i_co = i_arr[co_mask_b]; u_co = u_arr[co_mask_b]
    
    abs_metric_names = [
        "ben", "ben_weighted",
        "unique_items", "unique_clicked_items", "unique_btc_items", "unique_contacted_items",
    ]
    rel_metric_names = [
        "unique_items_rel", "unique_clicked_items_rel",
        "unique_btc_items_rel", "unique_contacted_items_rel",
    ]
    metric_names = abs_metric_names + rel_metric_names
    
    def _count_uniq_subset(i_subset, u_subset, group_vec, side):
        if len(i_subset) == 0:
            return 0
        rg = group_vec[u_subset]
        sel = (rg == side)
        if not sel.any():
            return 0
        c = np.bincount(i_subset[sel], minlength=n_items)
        return int((c > 0).sum())
    
    def compute_all_metrics(group_vec):
        row_group = group_vec[u_arr]
        mask_A = row_group == 0
        n_A = int((group_vec == 0).sum())
        n_B = n_users - n_A
        inv_nA = 1.0 / max(n_A, 1)
        inv_nB = 1.0 / max(n_B, 1)
        
        # BEN (in-place memory-efficient)
        c_A = np.bincount(i_arr[mask_A], minlength=n_items).astype(np.float64)
        c_B = c_total - c_A
        nlq_A = -np.log(np.maximum(c_A * inv_nA, eps))
        nlq_B = -np.log(np.maximum(c_B * inv_nB, eps))
        nlq_per_row = nlq_B[i_arr]
        nlq_per_row[mask_A] = nlq_A[i_arr[mask_A]]
        sum_nlq = np.bincount(u_arr, weights=nlq_per_row, minlength=n_users)
        np.multiply(nlq_per_row, w_arr, out=nlq_per_row)
        sum_wnlq = np.bincount(u_arr, weights=nlq_per_row, minlength=n_users)
        del nlq_per_row, nlq_A, nlq_B
        ben_u = sum_nlq * inv_k_u
        ben_w_u = sum_wnlq * inv_sum_w
        
        mA = group_vec == 0
        d_ben  = float(ben_u[~mA].mean()  - ben_u[mA].mean())
        d_benw = float(ben_w_u[~mA].mean() - ben_w_u[mA].mean())
        
        # unique_items: бесплатно из c_A/c_B
        nA_uni = int((c_A > 0).sum())
        nB_uni = int((c_B > 0).sum())
        d_uni = nB_uni - nA_uni
        
        # Sparse subsets
        nA_cl = _count_uniq_subset(i_cl, u_cl, group_vec, 0)
        nB_cl = _count_uniq_subset(i_cl, u_cl, group_vec, 1)
        d_clk = nB_cl - nA_cl
        nA_bt = _count_uniq_subset(i_bt, u_bt, group_vec, 0)
        nB_bt = _count_uniq_subset(i_bt, u_bt, group_vec, 1)
        d_btc = nB_bt - nA_bt
        nA_co = _count_uniq_subset(i_co, u_co, group_vec, 0)
        nB_co = _count_uniq_subset(i_co, u_co, group_vec, 1)
        d_con = nB_co - nA_co
        
        # Relative
        d_uni_rel = (nB_uni - nA_uni) / nA_uni if nA_uni > 0 else np.nan
        d_clk_rel = (nB_cl - nA_cl) / nA_cl if nA_cl > 0 else np.nan
        d_btc_rel = (nB_bt - nA_bt) / nA_bt if nA_bt > 0 else np.nan
        d_con_rel = (nB_co - nA_co) / nA_co if nA_co > 0 else np.nan
        
        return (d_ben, d_benw, d_uni, d_clk, d_btc, d_con,
                d_uni_rel, d_clk_rel, d_btc_rel, d_con_rel)
    
    obs = compute_all_metrics(group_vec)
    d_obs = dict(zip(metric_names, obs))
    
    # Уровни nC, nT (для отчёта rel %)
    unique_levels = {}
    for name, i_sub, u_sub in [
        ("unique_items", i_arr, u_arr),
        ("unique_clicked_items", i_cl, u_cl),
        ("unique_btc_items", i_bt, u_bt),
        ("unique_contacted_items", i_co, u_co),
    ]:
        unique_levels[name] = (
            _count_uniq_subset(i_sub, u_sub, group_vec, 0),
            _count_uniq_subset(i_sub, u_sub, group_vec, 1),
        )
    
    deltas = {name: np.empty(n_perm) for name in metric_names}
    group_p = group_vec.copy()
    for b in range(n_perm):
        rng.shuffle(group_p)
        ds = compute_all_metrics(group_p)
        for j, name in enumerate(metric_names):
            deltas[name][b] = ds[j]
    
    results = {}
    for name in metric_names:
        d = float(d_obs[name])
        arr = deltas[name]
        arr_clean = arr[~np.isnan(arr)]
        n_clean = len(arr_clean)
        mean_p = float(np.mean(arr_clean)) if n_clean > 0 else 0.0
        if n_clean == 0 or np.isnan(d):
            p, h0_lo, h0_hi = 1.0, np.nan, np.nan
        else:
            if d != 0:
                p = (np.sum(np.abs(arr_clean - mean_p) >= abs(d - mean_p)) + 1) / (n_clean + 1)
            else:
                p = 1.0
            h0_lo = float(np.percentile(arr_clean, 2.5))
            h0_hi = float(np.percentile(arr_clean, 97.5))
        entry = {
            "delta_obs": d,
            "p_value": float(min(p, 1.0)),
            "h0_lower": h0_lo,
            "h0_upper": h0_hi,
            "is_significant": (not np.isnan(d)) and (not np.isnan(h0_lo)) and ((d < h0_lo) or (d > h0_hi)),
        }
        base = name.replace("_rel", "")
        if base in unique_levels:
            nC, nT = unique_levels[base]
            entry["control_level"]   = int(nC)
            entry["treatment_level"] = int(nT)
            if nC > 0:
                entry["relative_change_pct"] = float((nT - nC) / nC * 100)
        results[name] = entry
    return results

## Ячейка 8 — Подготовка numpy-массивов

Конвертация pandas → numpy с правильными dtype для скорости:
- `cookie_id`, `item_id` → `int32` через `pd.factorize` (mapping в плотный диапазон 0..N)
- `position` → `float64` для DCG-весов
- `was_*` → `int8`

`pd.factorize` критичен для скорости: `np.bincount(cookie_id_int32)` работает в разы быстрее чем хеш-агрегация.

После этого исходный pandas освобождается через `del`/`gc.collect()`.


In [ ]:

t = time.time()
print(f"\nПодготовка numpy массивов...")

from pandas import factorize as _fact
u_arr = _fact(imp_pd["cookie_id"], sort=False)[0].astype(np.int32)
i_arr = _fact(imp_pd["item_id"],   sort=False)[0].astype(np.int32)
pos_arr = imp_pd["position"].values.astype(np.float64)
clicks_arr   = imp_pd["was_clicked"].values.astype(np.int8)
btc_arr      = imp_pd["was_btc"].values.astype(np.int8)
contacts_arr = imp_pd["was_contact"].values.astype(np.int8)

n_users_total = int(u_arr.max()) + 1
n_items_total = int(i_arr.max()) + 1

print(f"  n_users={n_users_total:,}, n_items={n_items_total:,}, rows={len(u_arr):,}")
print(f"  Time: {time.time()-t:.0f}s")

del imp_pd
import gc; gc.collect()

## Ячейка 9 — AA-цикл с multiprocessing + resume

**Главный цикл.** Каждая AA-итерация:
1. Случайно делим юзеров пополам с своим seed
2. Применяем `unified_permutation_test` на этой паре
3. Сохраняем результат в HDFS как `iter_NNNN.parquet`

### Параллелизация

- AA-итерации **полностью независимы** → embarrassingly parallel
- `multiprocessing.Pool(processes=N_WORKERS, context=fork)` — fork-CoW означает что numpy-массивы из родителя видны worker'ам через память без pickling
- Каждый worker возвращает только маленький dict (~1 KB после strip `_distrib`)
- Главный процесс собирает результаты через `imap_unordered` — streaming прогресс

### Resume

При старте проверяем какие `iter_NNNN.parquet` уже лежат в HDFS — пропускаем их. Считаем только недостающие.

### Resilient session

Каждые `PING_EVERY_AA` итераций — `ensure_spark_alive()`. Если YARN убил session — пересоздаём, продолжаем.

**Время на 200 итераций:** при N_WORKERS=6 на NUMA-bound железе ~4-5 часов реальных.


In [ ]:
#
# Каждая AA-итерация полностью независима: своя random seed → свой 50/50
# split юзеров → один пермутационный тест. Embarrassingly parallel.
#
# Реализация:
#   1. Большие numpy-массивы (u_arr, i_arr, ...) — module-level globals.
#      multiprocessing.Pool(fork) даёт child'ам доступ к ним через CoW
#      без pickling.
#   2. Каждая завершённая AA-итерация СРАЗУ сохраняется на HDFS как parquet.
#      При перезапуске kernel мы просто пропускаем уже сделанные iter_idx.
#   3. Spark session пересоздаётся автоматически если YARN убил предыдущую
#      (см. ensure_spark_alive() в Ячейке 1).
#   4. После каждых PING_EVERY_AA итераций — keep-alive ping. Если ping
#      зафейлился — пересоздаём session, продолжаем дальше.

import multiprocessing as mp
from multiprocessing import get_context

# === Setup кэша AA-итераций ===
aa_cache_dir = f"{cache_dir}/aa_iters_n{N_AA}_p{N_PERM}_a{int(ALPHA*1000)}"
print(f"\n=== AA cache ===")
print(f"  Directory: {aa_cache_dir}")

def _aa_iter_path(iter_idx):
    return f"{aa_cache_dir}/iter_{iter_idx:04d}.parquet"

def _save_aa_result(iter_idx, result_dict):
    """Сериализуем dict с результатами одной AA-итерации в parquet на HDFS.
    Single-row DataFrame с json-payload — компактно и быстро."""
    payload = json.dumps(result_dict, default=str)
    df = spark.createDataFrame([(iter_idx, payload)], ["iter_idx", "payload"])
    df.coalesce(1).write.mode("overwrite").parquet(_aa_iter_path(iter_idx))

def _load_aa_result(iter_idx):
    """Читаем сохранённую AA-итерацию обратно."""
    df = spark.read.parquet(_aa_iter_path(iter_idx)).collect()
    payload = df[0]["payload"]
    return json.loads(payload)

def _list_completed_iters():
    """Какие iter_idx уже посчитаны и лежат в кэше."""
    if not _hdfs_path_exists(aa_cache_dir):
        return set()
    try:
        jvm = spark._jvm
        path_obj = jvm.org.apache.hadoop.fs.Path(aa_cache_dir)
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI.create(aa_cache_dir),
                                                      spark._jsc.hadoopConfiguration())
        statuses = fs.listStatus(path_obj)
        completed = set()
        for s in statuses:
            name = s.getPath().getName()  # iter_NNNN.parquet
            if name.startswith("iter_") and name.endswith(".parquet"):
                try:
                    idx = int(name[5:-8])  # "iter_0042.parquet" -> 42
                    completed.add(idx)
                except ValueError:
                    pass
        return completed
    except Exception as e:
        print(f"  [aa_cache] _list_completed_iters error: {e}")
        return set()


# Глобальные ссылки для fork-CoW
_GLOBALS = {
    "u_arr": u_arr, "i_arr": i_arr, "pos_arr": pos_arr,
    "clicks_arr": clicks_arr, "btc_arr": btc_arr, "contacts_arr": contacts_arr,
    "n_users": n_users_total, "n_items": n_items_total,
    "n_perm": N_PERM,
}


def _aa_worker(args):
    """Один AA-iter. (iter_idx, seed) -> dict."""
    iter_idx, seed = args
    rng_split = np.random.default_rng(seed)
    n_users = _GLOBALS["n_users"]
    perm = rng_split.permutation(n_users)
    half = n_users // 2
    group_vec = np.empty(n_users, dtype=np.int8)
    group_vec[perm[:half]] = 0
    group_vec[perm[half:]] = 1
    res = unified_permutation_test(
        _GLOBALS["u_arr"], _GLOBALS["i_arr"], _GLOBALS["pos_arr"],
        _GLOBALS["clicks_arr"], _GLOBALS["btc_arr"], _GLOBALS["contacts_arr"],
        n_users, _GLOBALS["n_items"], group_vec,
        n_perm=_GLOBALS["n_perm"], seed=seed + 11,
    )
    return {"iter": iter_idx, "seed": seed, "results": res}


# === Resume: какие iters уже посчитаны? ===
completed_iters = _list_completed_iters()
print(f"  Уже посчитано: {len(completed_iters)}/{N_AA} iters")
if completed_iters:
    print(f"  Резюм с iter_idx >= {max(completed_iters) + 1 if completed_iters else 0}")

# Восстанавливаем уже посчитанные результаты в all_results
all_results = [None] * N_AA
if completed_iters:
    print(f"  Загружаем {len(completed_iters)} сохранённых результатов...", flush=True)
    t_load = time.time()
    for iter_idx in sorted(completed_iters):
        try:
            all_results[iter_idx] = _load_aa_result(iter_idx)
        except Exception as e:
            print(f"  Ошибка загрузки iter {iter_idx}: {e}, пересчитаем")
            completed_iters.discard(iter_idx)
            all_results[iter_idx] = None
    print(f"  Загружено за {time.time()-t_load:.0f}s")

remaining_tasks = [(i, SEED_BASE + i * 13) for i in range(N_AA) if i not in completed_iters]
print(f"  Осталось посчитать: {len(remaining_tasks)} iters")

if not remaining_tasks:
    print(f"  ВСЕ AA-итерации уже в кэше — пропускаем цикл, идём в анализ.")
else:
    print(f"\n=== Запуск AA: {N_WORKERS} parallel workers ===\n", flush=True)
    t_aa_start = time.time()
    
    def _process_streaming(stream_iter):
        """Обрабатываем результаты worker'ов по мере поступления.
        Saves каждый iter в HDFS, поддерживает счётчик прогресса."""
        completed_in_run = 0
        total_to_do = len(remaining_tasks)
        for res in stream_iter:
            iter_idx = res["iter"]
            all_results[iter_idx] = res
            # Save на HDFS — атомарно для этого iter
            try:
                _save_aa_result(iter_idx, res)
            except Exception as e:
                print(f"  [WARN] save iter {iter_idx} failed: {e}", flush=True)
                # Пробуем восстановить session и сохранить ещё раз
                ensure_spark_alive()
                try:
                    _save_aa_result(iter_idx, res)
                    print(f"  [recovered] save iter {iter_idx} after session reset", flush=True)
                except Exception as e2:
                    print(f"  [FATAL] второй save тоже упал: {e2}", flush=True)
                    # iter в all_results всё равно есть, продолжаем
            
            completed_in_run += 1
            el = time.time() - t_aa_start
            eta = el / completed_in_run * (total_to_do - completed_in_run)
            total_done = sum(1 for r in all_results if r is not None)
            print(f"  AA {total_done}/{N_AA} (run {completed_in_run}/{total_to_do}, "
                  f"iter={iter_idx}): elapsed {el:.0f}s, ETA {eta/60:.1f}min", flush=True)
            
            # Keep-alive: и пингуем session, и пересоздаём если умерла
            if completed_in_run % PING_EVERY_AA == 0:
                _spark, recreated = ensure_spark_alive()
                if recreated:
                    # globals для следующих save обновятся через _SPARK_TASK
                    print(f"    [resilient] session was recreated", flush=True)
    
    if N_WORKERS > 1:
        ctx = get_context("fork")
        with ctx.Pool(processes=N_WORKERS) as pool:
            _process_streaming(pool.imap_unordered(_aa_worker, remaining_tasks, chunksize=1))
    else:
        # Sequential fallback
        print(f"  N_WORKERS=1: sequential mode", flush=True)
        _process_streaming(_aa_worker(t) for t in remaining_tasks)
    
    print(f"\n=== AA готов за {(time.time()-t_aa_start)/60:.1f} минут "
          f"({len(remaining_tasks)} новых iters, {len(completed_iters)} из кэша) ===")

# Финальная валидация
missing = [i for i, r in enumerate(all_results) if r is None]
if missing:
    raise RuntimeError(f"AA-итерации не завершены: {missing[:10]}{'...' if len(missing)>10 else ''}")
print(f"\nВсего: {N_AA} AA-итераций готовы.")

## Ячейка 10 — Агрегация: 5 тестов калибровки

**Что происходит:** для каждой из 10 метрик прогоняем 5 тестов калибровки.

### Метрики калибровки

| Тест | H₀ | Что говорит |
|---|---|---|
| Wilson 95% CI | p_true = α | Корректный CI для биномиальной пропорции при малых n |
| Clopper-Pearson | p_true = α | Точный CI через Beta-распределение |
| Биномиальный тест | p_emp = α | Двусторонний тест, чувствителен к небольшим отклонениям |
| KS-test | p ~ Uniform(0,1) | Общая равномерность всего распределения p-value |
| Anderson-Darling | p ~ Uniform(0,1) | Равномерность с акцентом на хвосты (важно для α!) |

### Vердикт

«Калибровка OK» если **все 5 тестов прошли**. Если хоть один провалился — копаем что не так с метрикой.

### Почему 5 тестов, а не один

- Wilson/CP смотрят только на одну точку — α. Могут быть «ОК» при сильно несправедливом распределении.
- KS смотрит на всё распределение, но не очень чувствителен к хвостам.
- AD более чувствителен в хвостах — что важно потому что нас интересует именно поведение около α.
- Биномиальный тест даёт численный p-value на главную гипотезу.

Все 5 закрывают разные слабые места. Если все 5 прошли — калибровка действительно надёжная.


In [ ]:

print(f"\n{'='*92}")
print(f"АГРЕГИРОВАННЫЕ РЕЗУЛЬТАТЫ ({N_AA} AA-итераций × {N_PERM} perm, α={ALPHA})")
print(f"{'='*92}")

# ============================================================
# Стат-тесты для калибровки p-value
# ============================================================
# Под H0 ожидаем что:
#   1. Type I error = α (в данном случае 5%) — биномиальный тест
#   2. p-value uniform на [0,1] — KS-тест
#   3. Распределение в хвосте около α без аномалий — Anderson-Darling
#
# Корректные методы:
#   - Wilson score CI: правильный CI для биномиальной пропорции
#     (Wald [p̂±1.96·SE] плох для малых p̂ и малых N — может включать <0)
#   - Clopper-Pearson: точный CI на основе Beta-распределения
#     (более консервативный чем Wilson, считается "ground truth")
#   - Биномиальный тест H0: p = α
#   - KS-тест на uniform(0,1) — общая uniformity всего распределения
#   - Anderson-Darling — более чувствителен в хвостах распределения,
#     что важно для α (мы тестируем именно хвост p<0.05)

def wilson_ci(k, n, conf=0.95):
    """Wilson score interval. Корректнее Wald'а при малых k или n."""
    if n == 0:
        return (0.0, 1.0)
    z = sps.norm.ppf(1 - (1 - conf) / 2)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    halfwidth = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, centre - halfwidth), min(1.0, centre + halfwidth))


def clopper_pearson(k, n, conf=0.95):
    """Точный CI через Beta-распределение. "Ground truth"."""
    if n == 0:
        return (0.0, 1.0)
    alpha_lvl = 1 - conf
    lo = sps.beta.ppf(alpha_lvl/2, k, n-k+1) if k > 0 else 0.0
    hi = sps.beta.ppf(1 - alpha_lvl/2, k+1, n-k) if k < n else 1.0
    return (float(lo), float(hi))


def evaluate_full(p_values, alpha=ALPHA):
    """Полная стат-проверка калибровки p-value."""
    p = np.array(p_values)
    n = len(p)
    k = int((p < alpha).sum())
    type_I = k / n
    
    # Wilson и Clopper-Pearson
    wilson = wilson_ci(k, n, conf=0.95)
    cp = clopper_pearson(k, n, conf=0.95)
    
    # Биномиальный тест: H0 p = α (двусторонний)
    binom_p = sps.binomtest(k, n, alpha, alternative="two-sided").pvalue
    
    # KS на uniform(0,1)
    ks_stat, ks_p = sps.kstest(p, "uniform")
    
    # Anderson-Darling (более чувствителен к хвостам — что важно для α)
    # scipy реализация требует distribution name; для uniform используем
    # uniform-аппроксимацию через ks или собственный AD-стат
    # Анализ AD: мы можем сравнить p-value с U(0,1) через AD
    try:
        # Через ad-on-uniform via scipy.stats.anderson_ksamp с ref uniform
        # Проще — через cdf-based statistic
        srt = np.sort(p)
        idx = np.arange(1, n + 1)
        ad_stat = -n - (1.0 / n) * np.sum(
            (2 * idx - 1) * (np.log(srt + 1e-12) + np.log(1 - srt[::-1] + 1e-12))
        )
        # Критическое значение AD для uniform: ≈2.49 при α=0.05 (большие N)
        ad_critical = 2.49
        ad_uniform_ok = ad_stat < ad_critical
    except Exception:
        ad_stat, ad_uniform_ok = np.nan, True
    
    return {
        "k": k, "n": n,
        "type_I_error": float(type_I),
        "wilson_ci": wilson,
        "cp_ci": cp,
        "binom_test_p": float(binom_p),    # H0: p = α
        "ks_p": float(ks_p),
        "ks_stat": float(ks_stat),
        "ad_stat": float(ad_stat),
        "ad_uniform_ok": bool(ad_uniform_ok),
        # Calibration "verdicts":
        "alpha_in_wilson": wilson[0] <= alpha <= wilson[1],
        "alpha_in_cp": cp[0] <= alpha <= cp[1],
        "binom_pass": binom_p > 0.05,
        "ks_uniform": ks_p > 0.05,
    }


metric_names = ["ben", "ben_weighted", "unique_items",
                 "unique_clicked_items", "unique_btc_items", "unique_contacted_items",
                 "unique_items_rel", "unique_clicked_items_rel",
                 "unique_btc_items_rel", "unique_contacted_items_rel"]

# ============================================================
# Таблица 1: Type I error + биномиальные CI
# ============================================================
print(f"\n--- ТАБЛИЦА 1: эмпирический Type I error и доверительные интервалы ---\n")
print(f"  {'Метрика':<32}{'k/N':>10}{'α_emp':>10}{'Wilson 95%':>16}"
      f"{'CP 95%':>16}{'Binom p':>10}")
print("  " + "-" * 92)

evs = {}
for m in metric_names:
    ps = [r["results"][m]["p_value"] for r in all_results]
    e = evaluate_full(ps)
    evs[m] = e
    label = m + (" *" if m.endswith("_rel") else "")
    print(f"  {label:<30}{e['k']:>4}/{e['n']:<4}"
          f"{e['type_I_error']*100:>9.1f}%"
          f"  [{e['wilson_ci'][0]*100:>4.1f},{e['wilson_ci'][1]*100:>4.1f}]"
          f"  [{e['cp_ci'][0]*100:>4.1f},{e['cp_ci'][1]*100:>4.1f}]"
          f"{e['binom_test_p']:>10.3f}")
print(f"\n  * — новые метрики v11 (relative change)")
print(f"  Binom p > 0.05: не отвергаем H0 что Type I = α (5%) — это хороший знак")

# ============================================================
# Таблица 2: uniformity p-value (KS + AD)
# ============================================================
print(f"\n--- ТАБЛИЦА 2: проверка uniformity p-value на [0,1] ---")
print(f"  H0: p ~ Uniform(0,1). Большой p-value KS/AD = калибровка ОК.\n")
print(f"  {'Метрика':<32}{'KS stat':>10}{'KS p':>10}{'AD stat':>10}{'AD ok?':>10}")
print("  " + "-" * 80)
for m in metric_names:
    e = evs[m]
    label = m + (" *" if m.endswith("_rel") else "")
    ad_flag = "✓" if e["ad_uniform_ok"] else "⚠"
    print(f"  {label:<32}{e['ks_stat']:>10.3f}{e['ks_p']:>10.3f}"
          f"{e['ad_stat']:>10.3f}{ad_flag:>10}")
print(f"\n  D-критическое для KS (N={N_AA}, α=0.05): {1.36/np.sqrt(N_AA):.3f}")
print(f"  AD-критическое для uniform: ~2.49 (большие N)")

# ============================================================
# Таблица 3: финальный verdict по всем тестам
# ============================================================
print(f"\n--- ТАБЛИЦА 3: финальный verdict по 4 тестам калибровки ---\n")
print(f"  Метрика — α_in_Wilson | α_in_CP | Binom_pass | KS_uniform | AD_uniform")
print("  " + "-" * 88)
for m in metric_names:
    e = evs[m]
    flags = [
        ("✓" if e["alpha_in_wilson"] else "✗"),
        ("✓" if e["alpha_in_cp"] else "✗"),
        ("✓" if e["binom_pass"] else "✗"),
        ("✓" if e["ks_uniform"] else "✗"),
        ("✓" if e["ad_uniform_ok"] else "✗"),
    ]
    label = m + (" *" if m.endswith("_rel") else "")
    overall = "✓ калибровка OK" if all(f == "✓" for f in flags) else "⚠ см. флаги"
    print(f"  {label:<30}     {flags[0]}        {flags[1]}        {flags[2]}"
          f"          {flags[3]}          {flags[4]}    {overall}")

# ============================================================
# Sig rate via H0 quantile (decision rule)
# ============================================================
print(f"\n--- ТАБЛИЦА 4: rejection rate через H0-квантили (decision rule) ---")
print(f"  Это альтернативный путь: эффект значим если delta_obs за пределами")
print(f"  perm-distribution квантилей [α/2, 1-α/2]. Должно быть ≈ {ALPHA*100:.0f}%.\n")
for m in metric_names:
    sigs = [r["results"][m]["is_significant"] for r in all_results]
    k_sig = sum(sigs)
    n_sig = len(sigs)
    rate = k_sig / n_sig
    wilson = wilson_ci(k_sig, n_sig)
    label = m + (" *" if m.endswith("_rel") else "")
    in_ci = "✓" if wilson[0] <= ALPHA <= wilson[1] else "⚠"
    print(f"  {label:<30}: {rate*100:5.1f}% "
          f" Wilson 95%=[{wilson[0]*100:4.1f},{wilson[1]*100:4.1f}]   {in_ci}")

# ============================================================
# Distributional check: Δ под H0 должно быть mean ≈ 0
# ============================================================
print(f"\n--- ТАБЛИЦА 5: распределение наблюдаемых Δ под H0 ---")
print(f"  Под H0 (random split) среднее Δ должно быть ≈ 0. Аномалии — bias в коде.\n")
for m in metric_names:
    ds = [r["results"][m]["delta_obs"] for r in all_results]
    label = m + (" *" if m.endswith("_rel") else "")
    print(f"  {label:<30}: mean={np.mean(ds):+.6f}  std={np.std(ds, ddof=1):.6f}  "
          f"|Δ|_max={max(abs(d) for d in ds):.6f}")

## Ячейка 11 — Сохранение результатов

Собираем `summary` со всеми параметрами AA + результатами 5 тестов на 10 метриках. Записываем в HDFS как `summary.json`.

После этой ячейки можно открывать любой ноутбук и читать готовые результаты для построения графиков, сравнений между прогонами с разными параметрами и т.д.

Iter-чекпоинты остаются в HDFS — при повторных запусках с теми же параметрами не пересчитываются.


In [ ]:

t = time.time()
summary = {
    "date_from":     DATE_FROM,
    "date_to":       DATE_TO,
    "logcats":       LOGCATS,
    "universe_sample_pct": UNIVERSE_SAMPLE_PCT,
    "top_k":         TOP_K,
    "n_aa":          N_AA,
    "n_perm":        N_PERM,
    "alpha":         ALPHA,
    "version":       "aa_v11_unified_permutation_with_rel",
    "n_users_universe": int(n_users_uni),
    "n_rows_universe":  int(n_rows),
    "all_results":   all_results,
}

try:
    spark.createDataFrame(
        [(json.dumps(summary, indent=2, ensure_ascii=False, default=str),)],
        ["summary"]
    ).coalesce(1).write.mode("overwrite").text(f"{PATH_RESULTS}/summary.json")
    print(f"Saved за {time.time()-t:.0f}s")
except Exception as e:
    print(f"Save err: {e}")

impressions.unpersist()
universe_users.unpersist()
if mcats_df is not None:
    mcats_df.unpersist()

print(f"\nГОТОВО. Анализ результатов смотри в Ячейке 10.")